# Nível 1: CVRP com escolha de porto seco (instâncias pequenas)

**Disciplina:** GCC118 Programação Matemática — Fase 2 (modelagem PLI e testes em solver).

Este notebook valida o modelo do Nível 1 descrito em `02_modelo_PLI_nivel1.md`:

- 1 produto (café beneficiado, em toneladas) e frota homogênea;
- pontos de coleta agregados (municípios/armazéns) e 2 ou 3 portos secos;
- **1 porto seco por rota** (o solver escolhe qual);
- objetivo: minimizar o custo logístico total, com toda a oferta obrigatoriamente coletada e entregue.

> **Aviso importante.** As instâncias abaixo são **sintéticas** (coordenadas em km num plano, com fator de desvio viário de 1,25) e todos os parâmetros de custo são **valores de exemplo**, usados só para validar o modelo e mostrar o trade-off. Para o trabalho final, troque-os por dados reais: matriz de distâncias via OSRM, diesel da ANP, produção da PAM/IBGE e portos secos da lista da Receita Federal.

Os símbolos seguem o arquivo de modelagem. As fórmulas usam a sintaxe \( \) e \[ \].

## 1. Dependências e código do modelo

In [ ]:
%pip install -q 'pulp==2.9.0' numpy pandas matplotlib

In [ ]:
%%writefile cafe_vrp.py

import math, time, random
import numpy as np
import pulp

def make_instance(n=8, m=2, seed=1, Q=30.0, K=6, Tmax=9.0, speed=60.0,
                  c_km=2.8, diesel=6.2, cons=0.40, f_veh=350.0,
                  F_term=(120.0, 60.0), tariff=(0.0, 0.0), cap_term=(1e9, 1e9),
                  s_pick=0.5, s_term=(0.5, 1.0), area=200.0):
    """Instancia SINTETICA (coordenadas em km no plano) para validar o modelo."""
    rng = random.Random(seed)
    depot = (area*0.5, area*0.1)
    pick = [(rng.uniform(0, area), rng.uniform(area*0.25, area)) for _ in range(n)]
    term = [(area*0.15, area*0.05), (area*0.9, area*0.0)][:m] if m <= 2 else \
           [(rng.uniform(0, area), rng.uniform(0, area*0.3)) for _ in range(m)]
    q = [round(rng.uniform(4, 14), 1) for _ in range(n)]
    coords = [depot] + pick + term
    N = len(coords)
    dist = np.array([[math.dist(a, b)*1.25 for b in coords] for a in coords])  # fator de desvio viario
    return dict(n=n, m=m, coords=coords, dist=dist, q=q, Q=Q, K=K, Tmax=Tmax, speed=speed,
                c_arc=dist*(c_km + diesel*cons), t_arc=dist/speed, f_veh=f_veh,
                F_term=list(F_term)[:m], tariff=list(tariff)[:m], cap_term=list(cap_term)[:m],
                s_pick=s_pick, s_term=list(s_term)[:m])

def solve(inst, policy="flexible", time_limit=60, msg=False, with_cap=False, solver_name='highs'):
    n, m = inst["n"], inst["m"]
    O = list(range(1, n+1)); D = list(range(n+1, n+m+1)); G = 0
    q = {i: inst["q"][i-1] for i in O}
    Q, K, Tmax = inst["Q"], inst["K"], inst["Tmax"]
    c, t = inst["c_arc"], inst["t_arc"]
    near = {i: min(D, key=lambda d: inst["dist"][i][d]) for i in O}

    A = [(G, i) for i in O]
    A += [(i, j) for i in O for j in O if i != j]
    A += [(i, d) for i in O for d in D]
    A += [(d, G) for d in D]
    if policy == "nearest":
        A = [(a, b) for (a, b) in A if not (a in O and b in D and b != near[a])
             and not (a in O and b in O and near[a] != near[b])]

    prob = pulp.LpProblem("CVRP_multiterminal", pulp.LpMinimize)
    x = {a: (pulp.LpVariable(f"x_{a[0]}_{a[1]}", lowBound=0, upBound=K, cat="Integer") if a[0] in D
             else pulp.LpVariable(f"x_{a[0]}_{a[1]}", cat="Binary")) for a in A}
    L = {i: pulp.LpVariable(f"L_{i}", lowBound=0, upBound=Q) for i in O}
    T = {v: pulp.LpVariable(f"T_{v}", lowBound=0, upBound=Tmax) for v in O}
    inn = lambda v: [a for a in A if a[1] == v]
    out = lambda v: [a for a in A if a[0] == v]
    Fd = dict(zip(D, inst["F_term"])); td = dict(zip(D, inst["tariff"])); Cd = dict(zip(D, inst["cap_term"]))
    sd = dict(zip(D, inst["s_term"])); sp = inst["s_pick"]

    z = {}
    if with_cap or any(v > 0 for v in td.values()):
        for (i, d) in [a for a in A if a[0] in O and a[1] in D]:
            z[(i, d)] = pulp.LpVariable(f"z_{i}_{d}", lowBound=0, upBound=Q)
            prob += z[(i, d)] <= L[i]
            prob += z[(i, d)] <= Q * x[(i, d)]
            prob += z[(i, d)] >= L[i] - Q * (1 - x[(i, d)])

    prob += (pulp.lpSum(inst["f_veh"] * x[a] for a in A if a[0] == G)
             + pulp.lpSum(c[a[0]][a[1]] * x[a] for a in A)
             + pulp.lpSum(Fd[a[0]] * x[a] for a in A if a[0] in D)
             + pulp.lpSum(td[d] * z[(i, d)] for (i, d) in z))

    for i in O:
        prob += pulp.lpSum(x[a] for a in out(i)) == 1
        prob += pulp.lpSum(x[a] for a in inn(i)) == 1
    for d in D:
        prob += pulp.lpSum(x[a] for a in inn(d)) == pulp.lpSum(x[a] for a in out(d))
    prob += pulp.lpSum(x[a] for a in out(G)) == pulp.lpSum(x[a] for a in inn(G))
    prob += pulp.lpSum(x[a] for a in out(G)) <= K

    for i in O:
        prob += L[i] >= q[i] + 0
        prob += L[i] <= Q
    for (i, j) in [a for a in A if a[0] in O and a[1] in O]:
        prob += L[j] >= L[i] + q[j] - (Q + q[j]) * (1 - x[(i, j)])
        prob += L[j] <= L[i] + q[j] + Q * (1 - x[(i, j)])
    for i in O:
        if (G, i) in x:
            prob += L[i] <= q[i] + Q * (1 - x[(G, i)])
            prob += L[i] >= q[i] - Q * (1 - x[(G, i)])

    for i in O:
        if (G, i) in x:
            prob += T[i] >= t[G][i] + sp - Tmax * (1 - x[(G, i)])
    for (i, j) in [a for a in A if a[0] in O and a[1] in O]:
        prob += T[j] >= T[i] + t[i][j] + sp - (Tmax + t[i][j] + sp) * (1 - x[(i, j)])
    for (i, d) in [a for a in A if a[0] in O and a[1] in D]:
        tail = t[i][d] + sd[d] + t[d][G]
        prob += T[i] + tail <= Tmax + (Tmax + tail) * (1 - x[(i, d)])

    if with_cap:
        for d in D:
            prob += pulp.lpSum(z[(i, dd)] for (i, dd) in z if dd == d) <= Cd[d]

    solver = (pulp.HiGHS(msg=msg, timeLimit=time_limit) if solver_name=='highs' else pulp.PULP_CBC_CMD(msg=msg, timeLimit=time_limit))
    t0 = time.time(); prob.solve(solver); el = time.time() - t0
    status = pulp.LpStatus[prob.status]
    if prob.status != 1 and prob.objective is None:
        return dict(status=status)
    succ = {}
    for a in A:
        if x[a].value() is not None and x[a].value() > 0.5:
            succ.setdefault(a[0], []).append(a[1])
    routes = []
    for first in succ.get(G, []):
        r = [G, first]
        while r[-1] != G:
            r.append(succ[r[-1]][0])
        routes.append(r)
    obj = pulp.value(prob.objective)
    return dict(status=status, obj=obj, time=el, hit_limit=el >= 0.97*time_limit, routes=routes, n_vars=len(prob.variables()),
                n_cons=len(prob.constraints), n_bin=len([a for a in A if a[0] not in D]))

def route_cost(inst, r):
    n = inst["n"]; D0 = n + 1
    d = r[-2]
    cost = inst["f_veh"] + sum(inst["c_arc"][a][b] for a, b in zip(r[:-1], r[1:])) + inst["F_term"][d - D0]
    return cost


def plot_routes(inst, res, title="", ax=None):
    import matplotlib.pyplot as plt
    n = inst["n"]; coords = inst["coords"]
    ax = ax or plt.subplots(figsize=(6, 6))[1]
    cols = plt.cm.tab10.colors
    for k, r in enumerate(res["routes"]):
        xs = [coords[v][0] for v in r]; ys = [coords[v][1] for v in r]
        ax.plot(xs, ys, "-o", color=cols[k % 10], lw=1.6, ms=4, label=f"rota {k+1}")
    ax.scatter(*coords[0], marker="s", s=90, c="k", zorder=5, label="garagem")
    for d in range(n+1, n+inst["m"]+1):
        ax.scatter(*coords[d], marker="^", s=130, c="red", zorder=5)
        ax.annotate(f"T{d-n}", coords[d], textcoords="offset points", xytext=(6, 6), color="red")
    for i in range(1, n+1):
        ax.annotate(f"{i} ({inst['q'][i-1]}t)", coords[i], textcoords="offset points", xytext=(4, 4), fontsize=7)
    ax.set_title(title); ax.set_xlabel("km"); ax.set_ylabel("km"); ax.legend(fontsize=7)
    return ax


In [ ]:
import importlib, cafe_vrp
importlib.reload(cafe_vrp)
from cafe_vrp import make_instance, solve, plot_routes
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.width', 140)

## 2. Instância mínima (6 coletas, 2 portos secos)

Nós: `0` = garagem; `1..n` = coletas; `n+1..n+m` = portos secos. Parâmetros de exemplo:

| Parâmetro | Valor | Significado |
|---|---|---|
| \(Q\) | 30 t | capacidade do caminhão |
| \(K\) | 6 | frota máxima |
| \(T_{max}\) | 12 h | duração máxima da rota (viagem + serviço) |
| \(f\) | R$ 350 | custo fixo por veículo usado |
| \(F_d\) | R$ 120 (T1) e R$ 60 (T2) | custo fixo por entrada no terminal |
| custo por km | 2,80 + diesel 6,20 × consumo 0,40 L/km | R$/km |

In [ ]:
inst6 = make_instance(n=6, m=2, seed=1, area=120, Tmax=12, K=6)
dados = pd.DataFrame({'coleta': range(1, 7), 'oferta_t': inst6['q'],
                      'x_km': [round(c[0], 1) for c in inst6['coords'][1:7]],
                      'y_km': [round(c[1], 1) for c in inst6['coords'][1:7]]})
display(dados)
print('Oferta total (t):', round(sum(inst6['q']), 1), '| capacidade Q =', inst6['Q'])

In [ ]:
res6 = solve(inst6, 'flexible', time_limit=30)
print(res6['status'], '| custo total = R$', round(res6['obj'], 2), '| tempo =', round(res6['time'], 2), 's')
print('Rotas:', res6['routes'])
plot_routes(inst6, res6, 'Instância n=6: solução ótima')
plt.show()

## 3. Trade-off: escolher o porto seco ou usar sempre o mais próximo?

Duas políticas, resolvidas com **o mesmo modelo**:

- **flexível:** o solver escolhe rota e porto seco de cada rota;
- **mais próximo:** cada coleta só pode ser entregue no porto seco geograficamente mais próximo dela, e uma rota só agrupa coletas com o mesmo porto mais próximo. A regra é implementada removendo arcos do conjunto \(A\).

In [ ]:
def detalhar(inst, res):
    n, D0 = inst['n'], inst['n'] + 1
    veic = inst['f_veh'] * len(res['routes'])
    desloc = sum(inst['c_arc'][a][b] for r in res['routes'] for a, b in zip(r[:-1], r[1:]))
    term = sum(inst['F_term'][r[-2] - D0] for r in res['routes'])
    km = sum(inst['dist'][a][b] for r in res['routes'] for a, b in zip(r[:-1], r[1:]))
    horas = sum(inst['t_arc'][a][b] for r in res['routes'] for a, b in zip(r[:-1], r[1:]))
    return dict(rotas=len(res['routes']), km=round(km, 1), horas_viagem=round(horas, 1),
                custo_veiculos=round(veic, 1), custo_deslocamento=round(desloc, 1),
                custo_terminais=round(term, 1), custo_total=round(res['obj'], 1))

inst8 = make_instance(n=8, m=2, seed=1, area=120, Tmax=12, K=6)
out, resd = {}, {}
for pol in ['flexible', 'nearest']:
    resd[pol] = solve(inst8, pol, time_limit=60)
    out[pol] = detalhar(inst8, resd[pol])
comp = pd.DataFrame(out).T
comp['economia_%'] = ''
comp.loc['flexible', 'economia_%'] = round(100 * (1 - comp.loc['flexible', 'custo_total'] / comp.loc['nearest', 'custo_total']), 1)
display(comp)
fig, axs = plt.subplots(1, 2, figsize=(12, 5.5))
plot_routes(inst8, resd['nearest'], 'Política: porto mais próximo', axs[0])
plot_routes(inst8, resd['flexible'], 'Política: porto flexível (ótimo)', axs[1])
plt.tight_layout(); plt.show()

**Leitura do resultado.** A política do porto mais próximo força rotas separadas por região e acaba acionando mais veículos. A política flexível consolida cargas e paga o custo do desvio quando isso evita um veículo e uma entrada de terminal a mais. Isso é exatamente o trade-off: o modelo compara \(f\), \(F_d\) e o custo dos arcos, sem uma regra fixa.

## 4. Sensibilidade a custos de exemplo (diesel e custo fixo do terminal)

Os preços abaixo são cenários ilustrativos. No trabalho final, use percentis da série histórica de diesel da ANP (baixo, base e alto).

In [ ]:
linhas = []
for diesel in [4.5, 6.2, 8.0]:
    for F in [(120, 60), (60, 120), (200, 20)]:
        inst = make_instance(n=7, m=2, seed=3, area=120, Tmax=12, K=6, diesel=diesel, F_term=F)
        rf = solve(inst, 'flexible', time_limit=30); rn = solve(inst, 'nearest', time_limit=30)
        linhas.append(dict(diesel=diesel, F_T1=F[0], F_T2=F[1], custo_flex=round(rf['obj'], 1), custo_prox=round(rn['obj'], 1),
                           economia_pct=round(100 * (1 - rf['obj'] / rn['obj']), 1), rotas_flex=len(rf['routes']), rotas_prox=len(rn['routes'])))
sens = pd.DataFrame(linhas); display(sens)

## 5. Escalabilidade do modelo exato

A Fase 2 pede tempo, gap e número de variáveis/restrições em instâncias crescentes. O solver é CBC, com limite de 20 s por instância; quando o limite é atingido, o resultado é a melhor solução encontrada, sem prova de otimalidade.

In [ ]:
linhas = []
for n in [4, 5, 6, 7, 8, 9, 10]:
    inst = make_instance(n=n, m=2, seed=1, area=120, Tmax=12, K=6)
    r = solve(inst, 'flexible', time_limit=20)
    linhas.append(dict(n_coletas=n, binarias=r['n_bin'], variaveis=r['n_vars'], restricoes=r['n_cons'],
                       custo=round(r['obj'], 1), tempo_s=round(r['time'], 2), parou_no_limite=r['hit_limit'], rotas=len(r['routes'])))
esc = pd.DataFrame(linhas); display(esc)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(esc['n_coletas'], esc['tempo_s'], 'o-'); ax.axhline(20, ls='--', c='gray')
ax.set_xlabel('coletas (n)'); ax.set_ylabel('tempo de solução (s)'); ax.set_title('Tempo do modelo exato (CBC)')
plt.tight_layout(); plt.show()
esc.to_csv('escalabilidade_nivel1.csv', index=False)

## 6. Extensão rápida do Nível 2 (tarifa por tonelada e capacidade do terminal)

O mesmo código aceita `tariff` (R$/t por terminal) e `cap_term` (t por terminal no horizonte) com `with_cap=True`. A carga entregue em cada terminal é medida pela variável \(z_{id}\) (ver `02_modelo_PLI_nivel1.md`, seção de extensão).

In [ ]:
inst2 = make_instance(n=8, m=2, seed=1, area=120, Tmax=12, K=6, tariff=(14.0, 9.0), cap_term=(45.0, 45.0))
r2 = solve(inst2, 'flexible', time_limit=60, with_cap=True)
n, D0 = inst2['n'], inst2['n'] + 1
entregue = {f'T{d-n}': 0.0 for d in range(D0, D0 + inst2['m'])}
for r in r2['routes']:
    entregue[f'T{r[-2]-n}'] += sum(inst2['q'][v-1] for v in r[1:-2])
print(r2['status'], 'custo = R$', round(r2['obj'], 1), '| rotas:', r2['routes'])
print('Toneladas entregues por terminal:', {k: round(v, 1) for k, v in entregue.items()}, '| capacidade 45 t cada')

## 7. Próximos passos

1. Trocar as instâncias sintéticas por dados reais (matriz OSRM, diesel ANP, produção PAM/IBGE, portos secos da Receita Federal).
2. Fase 3: heurística construtiva (economias de Clarke-Wright com escolha do terminal) e busca local (*relocate*, *swap*, *2-opt*), comparada com o exato neste mesmo formato de tabela e gráfico.
3. Fase 4: cenários de diesel e demanda e discussão de incertezas.
4. Se sobrar tempo: janelas de tempo (Nível 3).